# Generate Synthetic Data
Creates a synthetic dataset matching the structure and column schema of `data_features.csv`.

In [1]:
import pandas as pd
import numpy as np
import uuid

# ── Configuration ─────────────────────────────────────────────────────────────
N_PARTICIPANTS = 200
SEED = 42
rng = np.random.default_rng(SEED)

# ── Column definitions ────────────────────────────────────────────────────────
test_types = ['ALT', 'Creatinine', 'Platelet']
display_formats = ['gradient', 'simple', 'block', 'table']
severities = ['further', 'slightly']

# Categorical value pools (from original data)
intention_values = ['Some form of immediate action', 'Willingness to wait']
gender_values = ['Female', 'Male']
ethnicity_values = ['Not Hispanic or Latino', 'Hispanic or Latino']
race_values = ['["White"]', '["Asian"]', '["Black or African American"]',
               '["White", "Asian"]', '["American Indian or Alaska Native"]']
education_values = ["Bachelor's degree", 'High school only',
                    "Master's degree", 'Some college', 'Doctoral degree']

# ── Generate participant IDs ──────────────────────────────────────────────────
participant_ids = [uuid.uuid4().hex[:24] for _ in range(N_PARTICIPANTS)]

# ── Assign each participant ONE display format (between-subjects) ─────────────
assigned_format = rng.choice(display_formats, size=N_PARTICIPANTS)

# ── Build rows ────────────────────────────────────────────────────────────────
rows = []
for i in range(N_PARTICIPANTS):
    row = {'participantId': participant_ids[i]}
    fmt = assigned_format[i]

    # Urgency scores (0 to 5 scale, 0.5 increments → diffs stay in -5 to 5)
    for tt in test_types:
        for sev in severities:
            for f in display_formats:
                col = f'{tt}_{sev}_{f}_urgency'
                if f == fmt:
                    row[col] = rng.choice(np.arange(0, 5.5, 0.5))
                else:
                    row[col] = np.nan

    # Urgency difference scores (further - slightly, range -5 to 5)
    for tt in test_types:
        for f in display_formats:
            col_diff = f'{tt}_{f}_urgency_diff'
            col_further  = f'{tt}_further_{f}_urgency'
            col_slightly = f'{tt}_slightly_{f}_urgency'
            if f == fmt:
                row[col_diff] = row[col_further] - row[col_slightly]
            else:
                row[col_diff] = np.nan

    # Intention columns (categorical)
    for tt in test_types:
        for sev in severities:
            for f in display_formats:
                col = f'{tt}_{sev}_{f}_intention'
                if f == fmt:
                    row[col] = rng.choice(intention_values)
                else:
                    row[col] = np.nan

    # Demographics & literacy scores
    row['graphical_literacy_score'] = int(rng.integers(1, 6))          # 1–5
    row['health_literacy']          = rng.choice(np.arange(1, 6, 0.5)) # 1–5
    row['subjective_numeracy']      = round(rng.uniform(1, 5), 3)      # 1–5
    row['familiarity']              = rng.choice(np.arange(1, 6, 1.0)) # 1–5
    row['age']                      = int(rng.integers(18, 75))
    row['gender']                   = rng.choice(gender_values)
    row['ethnicity']                = rng.choice(ethnicity_values)
    row['race']                     = rng.choice(race_values)
    row['education']                = rng.choice(education_values)
    row['prolific_id']              = row['participantId']

    rows.append(row)

# ── Assemble DataFrame with original column order ─────────────────────────────
original_cols = [
    'participantId',
    # urgency scores
    *[f'{tt}_{sev}_{f}_urgency'
      for tt in test_types for sev in severities for f in display_formats],
    # urgency diffs
    *[f'{tt}_{f}_urgency_diff'
      for tt in test_types for f in display_formats],
    # intention
    *[f'{tt}_{sev}_{f}_intention'
      for tt in test_types for sev in severities for f in display_formats],
    # demographics
    'graphical_literacy_score', 'health_literacy', 'subjective_numeracy',
    'familiarity', 'age', 'gender', 'ethnicity', 'race', 'education',
    'prolific_id',
]

df_synth = pd.DataFrame(rows, columns=original_cols)

# ── Save to CSV ───────────────────────────────────────────────────────────────
df_synth.to_csv('synthetic_data_features.csv', index=False)
print(f'Saved synthetic_data_features.csv  —  {df_synth.shape[0]} rows × {df_synth.shape[1]} cols')
df_synth.head()

Saved synthetic_data_features.csv  —  200 rows × 71 cols


,participantId,ALT_further_gradient_urgency,ALT_further_simple_urgency,ALT_further_block_urgency,ALT_further_table_urgency,ALT_slightly_gradient_urgency,ALT_slightly_simple_urgency,ALT_slightly_block_urgency,ALT_slightly_table_urgency,Creatinine_further_gradient_urgency,...,graphical_literacy_score,health_literacy,subjective_numeracy,familiarity,age,gender,ethnicity,race,education,prolific_id
0,ca3a6650b4cf444f931c7163,2.0,NaN,NaN,NaN,4.5,NaN,NaN,NaN,2.5,...,4,3.0,2.089,1.0,23,Female,Hispanic or Latino,"[""White""]",Master's degree,ca3a6650b4cf444f931c7163
1,fa3612fa9e6e490e85fa16bc,NaN,NaN,NaN,3.5,NaN,NaN,NaN,1.0,NaN,...,3,4.5,2.728,2.0,53,Female,Hispanic or Latino,"[""White""]",Some college,fa3612fa9e6e490e85fa16bc
2,8fa8c9a5940a460f81ec2958,NaN,NaN,3.0,NaN,NaN,NaN,0.0,NaN,NaN,...,4,1.5,3.351,4.0,27,Female,Hispanic or Latino,"[""American Indian or Alaska Native""]",Master's degree,8fa8c9a5940a460f81ec2958
3,a3a4e1e4c4b248d5b2a9dfdd,NaN,2.0,NaN,NaN,NaN,1.5,NaN,NaN,NaN,...,1,1.0,2.947,4.0,45,Female,Hispanic or Latino,"[""White""]",Master's degree,a3a4e1e4c4b248d5b2a9dfdd
4,5b0d589ef58a45998444756d,NaN,2.5,NaN,NaN,NaN,2.5,NaN,NaN,NaN,...,2,5.0,4.585,3.0,25,Female,Hispanic or Latino,"[""Asian""]",Bachelor's degree,5b0d589ef58a45998444756d
